# FindFire: from FIRMS hotspots to tracked fire zones and a replay

**Pipeline (MVP):** fetch hotspots → group into zones → track zones over time → show on an interactive map (one layer per 12 h step).

What this version does (based on how published work, e.g. the FEDS method, handles it):
- **12-hour steps**, matching the two daily satellite passes.
- **Cumulative zones:** a zone keeps every hotspot assigned to it since it started, so its area never shrinks just because a satellite missed a pass.
- **Tracking:** new hotspots join an existing zone if they are close to it, otherwise they start a new zone. Zones that grow into each other **merge** (the older ID wins).
- **Status:** `new` (first seen < 24 h ago), `active`, `inactive` (no detections for 3 days).
- **Algerian coordinates:** clustering uses the haversine metric (correct distances anywhere); area and perimeter are measured in the **UTM zone of each zone's centre** (29N to 32N), never in one fixed zone.
- **Two VIIRS satellites** (Suomi NPP and NOAA-20) observe within about an hour of each other, so a fire can be detected twice. Zone areas are unaffected (overlapping pixels merge), but `n_hotspots` and `frp_total_mw` count every detection from every satellite.

Set the key with `export FIRMS_MAP_KEY=...` before starting Jupyter. Without a key or network, the notebook uses **synthetic demo data** and says so.

In [ ]:
import os, io, json, time, pathlib, datetime as dt
from functools import lru_cache
import numpy as np, pandas as pd, requests
import matplotlib.pyplot as plt
from matplotlib.patches import Patch
import geopandas as gpd
from sklearn.cluster import DBSCAN
from sklearn.neighbors import BallTree
from shapely.geometry import box, mapping
import shapely
from shapely.ops import unary_union
from pyproj import Transformer
import folium                         # pip install folium

plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})

MAP_KEY = os.environ.get("FIRMS_MAP_KEY")
BBOX = (-2.0, 34.0, 8.5, 37.5)                  # west, south, east, north
SOURCES = ["VIIRS_SNPP_SP", "VIIRS_NOAA20_SP"]   # archive; use the *_NRT versions for recent data
START, END = dt.date(2021, 8, 8), dt.date(2021, 8, 14)
EVENT = "kabylie_2021_08"

STEP = pd.Timedelta(hours=12)                   # one step per satellite pass window
LINK_M = 1500                                   # hotspots closer than this belong together
ATTACH_M = 3000                                 # new detections this close to an existing zone join it (and zones this close merge)
CLOSE_M = 750                                   # closes small gaps inside a polygon
INACTIVE = pd.Timedelta(days=3)                 # no detections for this long -> inactive
NEW_FOR = pd.Timedelta(hours=24)
LOCAL = pd.Timedelta(hours=1)                   # Algeria is UTC+1 all year
EARTH_R = 6371008.8
STATUS_COL = {"new": "#e63946", "active": "#f4a261", "inactive": "#6c757d"}

## 1. Load data

In [ ]:
def fetch_chunk(source, start, days):
    w, s, e, n = BBOX
    url = f"https://firms.modaps.eosdis.nasa.gov/api/area/csv/{MAP_KEY}/{source}/{w},{s},{e},{n}/{days}/{start:%Y-%m-%d}"
    r = requests.get(url, timeout=60); r.raise_for_status()
    if not r.text.startswith("latitude"):
        raise RuntimeError(r.text[:200])
    return pd.read_csv(io.StringIO(r.text))

def fetch_firms(sources, start, end):
    """Download every source for the date range (5 days per request) and stack them.
    A source that fails is skipped with a warning; if all fail, raise."""
    frames = []
    for source in sources:
        try:
            cur = start
            while cur <= end:
                days = min(5, (end - cur).days + 1)
                part = fetch_chunk(source, cur, days); part["source"] = source
                frames.append(part); cur += dt.timedelta(days=days); time.sleep(1)
        except Exception as exc:
            print(f"warning: {source} skipped ({exc})")
    if not frames:
        raise RuntimeError("no data downloaded from any source")
    return pd.concat(frames, ignore_index=True).drop_duplicates()

def make_synthetic(seed=7):
    """Fake but plausible: 5 growing fires (two of them converge and merge), one that dies out,
    two passes a day, plus a few isolated false alarms."""
    rng = np.random.default_rng(seed)
    fires = [(36.70, 4.30, 0, 6), (36.72, 4.36, 2, 6), (36.75, 5.70, 1, 6),
             (35.40, 1.20, 2, 3), (36.30, 7.20, 3, 6)]          # lat, lon, first day, last day
    rows = []
    for lat0, lon0, d0, d1 in fires:
        for day in range(d0, d1 + 1):
            spread = 0.004 + 0.003 * (day - d0)
            for hhmm in (130, 1330):
                for _ in range(int(rng.integers(8, 20 + 15 * (day - d0)))):
                    dy, dx = np.clip(rng.normal(0, [spread, spread * 1.2]), [-2.2 * spread, -2.6 * spread], [2.2 * spread, 2.6 * spread])
                    rows.append(dict(
                        latitude=lat0 + dy, longitude=lon0 + dx,
                        acq_date=str(START + dt.timedelta(days=day)), acq_time=hhmm + int(rng.integers(0, 25)),
                        confidence=rng.choice(["n", "h", "l"], p=[0.5, 0.4, 0.1]), frp=float(rng.lognormal(1.6, 0.9)),
                        scan=float(rng.uniform(0.375, 0.7)), track=float(rng.uniform(0.375, 0.6)), satellite="N", source="SYNTHETIC"))
    for _ in range(10):
        rows.append(dict(latitude=rng.uniform(34.2, 37.3), longitude=rng.uniform(-1.8, 8.3),
                         acq_date=str(START + dt.timedelta(days=int(rng.integers(0, 7)))), acq_time=1330,
                         confidence="n", frp=float(rng.lognormal(0.8, 0.4)),
                         scan=0.5, track=0.45, satellite="N", source="SYNTHETIC"))
    return pd.DataFrame(rows)

def prepare(raw):
    d = raw.copy()
    d["acq_dt"] = pd.to_datetime(d["acq_date"] + " " + d["acq_time"].astype(int).astype(str).str.zfill(4),
                                 format="%Y-%m-%d %H%M", utc=True)
    d = d[d["confidence"].astype(str).str.lower() != "l"]          # drop low confidence
    for c in ("scan", "track"):                                    # pixel size in km (FIRMS columns)
        d[c] = d[c] if c in d else 0.375
    return d.sort_values("acq_dt").reset_index(drop=True)

DATA_SOURCE = "FIRMS"
try:
    if not MAP_KEY: raise RuntimeError("FIRMS_MAP_KEY not set")
    raw = fetch_firms(SOURCES, START, END)
except Exception as exc:
    print("Could not fetch FIRMS data:", exc)
    raw = make_synthetic(); DATA_SOURCE = "SYNTHETIC DEMO DATA"

df = prepare(raw)
print(f"DATA SOURCE: {DATA_SOURCE} | {len(df)} hotspots after removing low confidence")

## 2. Zone geometry in the right Algerian UTM zone
Algeria spans UTM zones 29N to 32N, so each zone is measured in the zone of its own centre. Each hotspot is a rectangle of its own pixel size (`scan` x `track`, which grows towards the swath edge).

In [ ]:
def utm_epsg(lon, lat):                        # Algeria is entirely in the northern hemisphere
    return 32600 + int((lon + 180) // 6) + 1

@lru_cache(maxsize=8)
def _tf(epsg):
    return (Transformer.from_crs(4326, epsg, always_xy=True).transform,
            Transformer.from_crs(epsg, 4326, always_xy=True).transform)

def zone_geometry(lons, lats, scan_km, track_km, epsg=None):
    """-> (polygon in lon/lat, area in ha, perimeter in km), measured in the local UTM zone."""
    epsg = epsg or utm_epsg(np.mean(lons), np.mean(lats))
    fwd, inv = _tf(epsg)
    sq = []
    for lo, la, s, t in zip(lons, lats, scan_km, track_km):
        x, y = fwd(lo, la)
        sq.append(box(x - s * 500, y - t * 500, x + s * 500, y + t * 500))
    poly = unary_union(sq).buffer(CLOSE_M).buffer(-CLOSE_M)       # close small gaps
    geo = shapely.transform(poly, lambda c: np.column_stack(inv(c[:, 0], c[:, 1])))
    return geo, poly.area / 1e4, poly.length / 1e3

## 3. Tracking
At each 12 h step: new hotspots (plus lone hotspots still waiting) are clustered. A cluster close to an active zone **joins it**; otherwise two or more hotspots **start a new zone**; a lone hotspot waits (it could be a flare). Zones that grow into each other **merge**.

In [ ]:
def _tree(latlon):
    return BallTree(np.radians(latlon), metric="haversine")

def status_of(z, t):
    if z["first"] > t - NEW_FOR: return "new"
    if z["last"] > t - INACTIVE: return "active"
    return "inactive"

def run_tracking(df):
    xy = np.c_[df.latitude.values, df.longitude.values]
    ts = df.acq_dt
    start = ts.min().floor("D")
    step_times = pd.date_range(start + STEP, ts.max().floor("12h") + STEP, freq="12h")
    r = LINK_M / EARTH_R          # clustering of new hotspots
    ra = ATTACH_M / EARTH_R       # joining / merging with existing zones
    zones, pending, merges, rows, hs_zone, geo_cache = {}, set(), [], [], {}, {}
    next_id = 1

    def active_ids(t):
        return [zid for zid, z in zones.items() if z["merged_into"] is None and z["last"] > t - INACTIVE]

    for t in step_times:
        t0 = t - STEP
        new_idx = set(df.index[(ts > t0) & (ts <= t)])
        pending = {i for i in pending if ts[i] > t - INACTIVE}
        cand = sorted(pending | new_idx)
        touched, still_pending = [], set()

        if cand:
            lab = DBSCAN(eps=r, min_samples=1, metric="haversine",
                         algorithm="ball_tree").fit_predict(np.radians(xy[cand]))
            act = active_ids(t)
            if act:
                pix = [(i, zid) for zid in act for i in zones[zid]["idx"]]
                pidx = np.array([p[0] for p in pix]); pz = np.array([p[1] for p in pix])
                tree = _tree(xy[pidx])
            for k in np.unique(lab):
                members = [cand[j] for j in np.where(lab == k)[0]]
                target = None
                if act:
                    dist, nn = tree.query(np.radians(xy[members]), k=1)
                    if dist.min() <= ra:
                        target = int(pz[nn[dist[:, 0].argmin(), 0]])
                if target is not None:                              # join an existing zone
                    z = zones[target]; z["idx"].update(members)
                    z["last"] = max(z["last"], ts[members].max()); touched += members
                elif len(members) >= 2:                             # start a new zone
                    zones[next_id] = dict(idx=set(members), first=ts[members].min(),
                                          last=ts[members].max(), merged_into=None)
                    touched += members; next_id += 1
                else:                                               # lone hotspot: wait
                    still_pending.update(members)
        pending = still_pending

        act = active_ids(t)                                        # merges
        if touched and len(act) > 1:
            pix = [(i, zid) for zid in act for i in zones[zid]["idx"]]
            pidx = np.array([p[0] for p in pix]); pz = np.array([p[1] for p in pix])
            own = {i: zid for i, zid in pix}
            tree = _tree(xy[pidx]); parent = {zid: zid for zid in act}
            def find(a):
                while parent[a] != a: parent[a] = parent[parent[a]]; a = parent[a]
                return a
            for i, nb in zip(touched, tree.query_radius(np.radians(xy[touched]), r=ra)):
                for j in nb:
                    a, b = find(own[i]), find(int(pz[j]))
                    if a != b: parent[b] = a
            groups = {}
            for zid in act: groups.setdefault(find(zid), []).append(zid)
            for g in groups.values():
                if len(g) > 1:
                    keep = min(g, key=lambda q: (zones[q]["first"], q))        # older ID wins
                    for s in g:
                        if s == keep: continue
                        zones[keep]["idx"] |= zones[s]["idx"]
                        zones[keep]["first"] = min(zones[keep]["first"], zones[s]["first"])
                        zones[keep]["last"] = max(zones[keep]["last"], zones[s]["last"])
                        zones[s]["merged_into"] = keep; merges.append((t, s, keep))

        step_map = {}
        for zid, z in zones.items():
            if z["merged_into"] is not None: continue
            idx = sorted(z["idx"]); key = (zid, len(idx))
            if key not in geo_cache:
                sub = df.loc[idx]
                geo_cache[key] = zone_geometry(sub.longitude.values, sub.latitude.values,
                                               sub.scan.values, sub.track.values)
            geom, area, per = geo_cache[key]
            sub = df.loc[idx]; step_sub = sub[(sub.acq_dt > t0) & (sub.acq_dt <= t)]
            rows.append(dict(step=t, zone_id=zid, status=status_of(z, t), area_ha=area, perimeter_km=per,
                             n_hotspots=len(sub), new_hotspots=len(step_sub), frp_new_mw=step_sub.frp.sum(),
                             frp_total_mw=sub.frp.sum(), first_detected=z["first"], last_detected=z["last"],
                             geometry=geom))
            for i in step_sub.index: step_map[i] = zid
        hs_zone[t] = step_map

    snap = (gpd.GeoDataFrame(rows, geometry="geometry", crs="EPSG:4326") if rows else
            gpd.GeoDataFrame({"step": [], "zone_id": [], "geometry": []}, geometry="geometry", crs="EPSG:4326"))
    return snap, zones, merges, step_times, hs_zone

### Quick tests (known answers)

In [ ]:
# 1) a 1 km square must be 100 ha
_, area, _ = zone_geometry([4.0], [36.0], [1.0], [1.0]); assert abs(area - 100) < 1, area
# 2) the same square measured in two neighbouring UTM zones must agree within 1 %
_, a31, _ = zone_geometry([5.9], [36.0], [1.0], [1.0], epsg=32631)
_, a32, _ = zone_geometry([5.9], [36.0], [1.0], [1.0], epsg=32632); assert abs(a31 - a32) / a31 < 0.01
# 3) two separate zones joined by new hotspots must merge into one (older ID wins)
def _pts(lons, when):
    return pd.DataFrame(dict(latitude=36.0, longitude=lons, acq_dt=pd.Timestamp(when, tz="UTC"),
                             frp=5.0, scan=0.4, track=0.4, confidence="n"))
t_df = pd.concat([_pts([4.0, 4.0025, 4.005], "2021-08-09 01:30"), _pts([4.04, 4.0425, 4.045], "2021-08-09 01:30"),
                  _pts([4.013, 4.026], "2021-08-09 13:30")], ignore_index=True).sort_values("acq_dt").reset_index(drop=True)
s_, z_, m_, *_ = run_tracking(t_df)
assert len(m_) == 1 and s_[s_.step == s_.step.max()].shape[0] == 1, "merge test failed"
# 4) a lone hotspot creates no zone
lone = _pts([4.0], "2021-08-09 01:30"); assert run_tracking(lone)[0].empty
print("all tests passed")

## 4. Run the pipeline

In [ ]:
snap, zones, merges, step_times, hs_zone = run_tracking(df)
final = snap[snap.step == step_times[-1]].copy()
print(f"{len(step_times)} steps of 12 h | {len(final)} zones at the end | merges: " +
      (", ".join(f"zone {s} into {k}" for _, s, k in merges) or "none"))
lone_left = len(df) - sum(len(z["idx"]) for z in zones.values() if z["merged_into"] is None)
print(f"hotspots not in any zone (isolated / possible false alarms): {lone_left}")
final.sort_values("area_ha", ascending=False)[["zone_id", "status", "area_ha", "perimeter_km", "n_hotspots",
    "frp_total_mw", "first_detected", "last_detected"]].round({"area_ha": 0, "perimeter_km": 1, "frp_total_mw": 0})

## 5. Interactive map (real basemap)
Zones are polygons (cumulative footprint, coloured by status); hotspots are dots coloured by the zone they belong to (grey = not in any zone). The layer control (top right) switches between a street map and satellite imagery, and has one layer per 12 h step: tick other steps to replay how the zones grew. Click a zone for its numbers. Tiles load from the internet (Esri; OpenStreetMap's own servers block requests from notebooks).

In [ ]:
PALETTE = ["#2a78d6", "#eb6834", "#1baf7a", "#eda100", "#e87ba4", "#008300", "#6250d6", "#e34948"]
NOTE = ("<div style='position:fixed;bottom:8px;left:50%;transform:translateX(-50%);z-index:9999;background:white;"
        "padding:3px 10px;border-radius:4px;font:12px sans-serif;border:1px solid #999'>"
        "Satellite detections (~375 m pixels). Areas are estimates. 'Inactive' means no detections, not necessarily out.</div>")

def zone_popup(z):
    return (f"<b>Zone #{z.zone_id}</b> ({z.status})<br>{z.area_ha:,.0f} ha (estimate) · {z.perimeter_km:.1f} km perimeter"
            f"<br>{z.n_hotspots} hotspots · FRP total {z.frp_total_mw:,.0f} MW"
            f"<br>first seen {(z.first_detected + LOCAL):%d %b %H:%M}, last {(z.last_detected + LOCAL):%d %b %H:%M} (UTC+1)")

def hs_color(zid):
    return "#898781" if zid is None else PALETTE[int(zid) % len(PALETTE)]

def base_map():
    b = snap.total_bounds                                           # minx, miny, maxx, maxy
    m = folium.Map(tiles=None, control_scale=True)
    folium.TileLayer(tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Street_Map/MapServer/tile/{z}/{y}/{x}",
                     attr="Tiles &copy; Esri", name="Street map", overlay=False).add_to(m)
    folium.TileLayer(tiles="https://server.arcgisonline.com/ArcGIS/rest/services/World_Imagery/MapServer/tile/{z}/{y}/{x}",
                     attr="Tiles &copy; Esri, Maxar, Earthstar Geographics", name="Satellite", overlay=False, show=False).add_to(m)
    m.fit_bounds([[b[1], b[0]], [b[3], b[2]]], padding=(40, 40))
    m.get_root().html.add_child(folium.Element(NOTE))
    return m

def zone_map(max_steps=40):
    m = base_map()
    steps = list(step_times)[-max_steps:]
    for i, t in enumerate(steps):
        fg = folium.FeatureGroup(name=f"{(t + LOCAL):%d %b %H:%M} (UTC+1)", show=(i == len(steps) - 1))
        for z in snap[snap.step == t].itertuples():
            folium.GeoJson(mapping(z.geometry), popup=folium.Popup(zone_popup(z), max_width=300),
                           tooltip=f"Zone #{z.zone_id}: {z.area_ha:,.0f} ha",
                           style_function=lambda _, c=STATUS_COL[z.status]: dict(color="black", weight=2,
                                                                                 fillColor=c, fillOpacity=0.45)).add_to(fg)
        for idx in df.index[(df.acq_dt > t - STEP) & (df.acq_dt <= t)]:
            r = df.loc[idx]; zid = hs_zone[t].get(idx)
            folium.CircleMarker([r.latitude, r.longitude], radius=3, color=hs_color(zid), fill=True, fill_opacity=0.85,
                                weight=0.5, tooltip=f"FRP {r.frp:.1f} MW, {(r.acq_dt + LOCAL):%H:%M} (UTC+1), conf {r.confidence}"
                                ).add_to(fg)
        fg.add_to(m)
    folium.LayerControl(collapsed=False).add_to(m)
    return m

zone_map()

## 6. How each zone grew
Outlines are the zone's footprint at successive 12 h steps (dark = early, bright = late); dots are its hotspots coloured by detection time.

In [ ]:
top = final.sort_values("area_ha", ascending=False).head(4)
cmap = plt.cm.viridis; n_steps = len(step_times)
step_idx = np.searchsorted(step_times.values, df.acq_dt.values.astype("datetime64[ns]"), side="left")
fig, axes = plt.subplots(2, 2, figsize=(11, 9))
for ax, (_, z) in zip(axes.ravel(), top.iterrows()):
    hist = snap[snap.zone_id == z.zone_id].sort_values("step")
    ks = [int(np.searchsorted(step_times.values, s.to_datetime64())) for s in hist.step]
    for k, g in zip(ks, hist.geometry):
        gpd.GeoSeries([g]).boundary.plot(ax=ax, color=cmap(k / max(n_steps - 1, 1)), linewidth=1.3)
    gpd.GeoSeries([z.geometry]).plot(ax=ax, facecolor=STATUS_COL[z.status], alpha=0.25, edgecolor="none")
    idx = sorted(zones[int(z.zone_id)]["idx"])
    ax.scatter(df.longitude[idx], df.latitude[idx], c=[cmap(k / max(n_steps - 1, 1)) for k in step_idx[idx]],
               s=7, zorder=3)
    minx, miny, maxx, maxy = z.geometry.bounds; pad = max(maxx - minx, maxy - miny) * 0.2 + 0.01
    ax.set_xlim(minx - pad, maxx + pad); ax.set_ylim(miny - pad, maxy + pad)
    ax.set_aspect(1 / np.cos(np.radians(36)))
    ax.set_title(f"zone #{z.zone_id}: {z.area_ha:,.0f} ha, {z.perimeter_km:.0f} km perimeter, {z.status}", fontsize=9)
    ax.tick_params(labelsize=7)
sm = plt.cm.ScalarMappable(cmap=cmap, norm=plt.Normalize(0, n_steps - 1)); sm.set_array([])
cb = fig.colorbar(sm, ax=axes, shrink=0.6, label="time step (12 h each)")
plt.show()

## 7. Zone history
Cumulative area per zone and new detections per step. Gaps in new detections are often just no satellite pass or cloud, which is why area is cumulative.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4.2))
for zid, g in snap.groupby("zone_id"):
    if g.area_ha.max() < 20: continue
    axes[0].plot(g.step + LOCAL, g.area_ha, marker="o", ms=3, label=f"#{zid}")
axes[0].set(title="Zone area (cumulative footprint, estimate)", ylabel="hectares"); axes[0].legend(fontsize=8, ncol=2)
newc = snap.groupby("step").new_hotspots.sum()
axes[1].bar(newc.index + LOCAL, newc.values, width=0.4, color="#d9632b")
axes[1].set(title="New hotspots per 12 h step (all zones)", ylabel="count")
for a in axes: a.tick_params(axis="x", rotation=30, labelsize=8)
plt.tight_layout(); plt.show()

## 8. Export for the web app
One JSON file per 12 h step, following the project's data contract. The map and replay in the web app only read these files.

In [ ]:
out = pathlib.Path("data") / EVENT; out.mkdir(parents=True, exist_ok=True)
iso = lambda x: pd.Timestamp(x).strftime("%Y-%m-%dT%H:%M:%SZ")
for t in step_times:
    zf = [dict(type="Feature", geometry=mapping(r.geometry), properties=dict(
              zone_id=int(r.zone_id), status=r.status, area_ha=round(r.area_ha, 1), perimeter_km=round(r.perimeter_km, 2),
              frp_total_mw=round(r.frp_total_mw, 1), n_hotspots=int(r.n_hotspots), new_hotspots=int(r.new_hotspots),
              frp_new_mw=round(r.frp_new_mw, 1), first_detected=iso(r.first_detected), last_detected=iso(r.last_detected)))
          for r in snap[snap.step == t].itertuples()]
    hs = df.loc[list(hs_zone[t])] if hs_zone[t] else df.iloc[0:0]
    hf = [dict(type="Feature", geometry=dict(type="Point", coordinates=[round(r.longitude, 5), round(r.latitude, 5)]),
               properties=dict(time=iso(r.acq_dt), frp_mw=round(r.frp, 1), zone_id=hs_zone[t][r.Index]))
          for r in hs.itertuples()]
    (out / f"{t:%Y%m%dT%H%M}.json").write_text(json.dumps(dict(
        time=iso(t), zones=dict(type="FeatureCollection", features=zf),
        hotspots=dict(type="FeatureCollection", features=hf))))
(out / "index.json").write_text(json.dumps(dict(event=EVENT, bbox=BBOX, step_hours=12, data=DATA_SOURCE, sources=SOURCES,
                                                steps=[iso(t) for t in step_times])))
print(f"wrote {len(step_times)} step files + index.json to {out}/  (data: {DATA_SOURCE})")